In [ ]:
!pip install boto3 python-dotenv

In [ ]:
# Load AWS credentials from .env file
import os
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Check environment variables
aws_access_key = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_key = os.getenv('AWS_SECRET_ACCESS_KEY')
aws_region = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')

# Bedrock API Example
Here's the complete working example:

In [2]:
import os
import sys
import boto3
from dotenv import load_dotenv
from botocore.exceptions import ClientError

# Remove Jupyter output limit
import IPython
from IPython.display import display, HTML

# Remove output length limit
IPython.display.max_repr_length = None
sys.stdout.reconfigure(encoding='utf-8') if hasattr(sys.stdout, 'reconfigure') else None

"""
This example demonstrates how to use Anthropic Claude 3.7 Sonnet's reasoning capability
with Amazon Bedrock. It shows how to:
- Load AWS credentials from .env file using python-dotenv
- Set up the Amazon Bedrock runtime client
- Create a message
- Configure reasoning parameters
- Send a request with reasoning enabled
- Process both the reasoning output and final response
"""

def reasoning_example():

    # Load environment variables from .env file
    load_dotenv()
    
    # Get AWS region from environment variable or use default
    aws_region = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')
    
    # Create the Amazon Bedrock runtime client
    # Credentials are automatically loaded from environment variables
    client = boto3.client("bedrock-runtime", region_name=aws_region)

    # Specify the model ID. For the latest available models, see:
    # https://docs.aws.amazon.com/bedrock/latest/userguide/models-supported.html
    model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

    # Create the message with the user's prompt
    user_message = "What are enterprise business problems that can be solved by building agents with reasoning models?"
    conversation = [
        {
            "role": "user",
            "content": [{"text": user_message}],
        }
    ]

    # Configure reasoning parameters with a 2000 token budget
    reasoning_config = {
        "thinking": {
            "type": "enabled",
            "budget_tokens": 2000
        }
    }

    # Send message and reasoning configuration to the model
    response = client.converse(
        modelId=model_id,
        messages=conversation,
        additionalModelRequestFields=reasoning_config
    )

    # Extract the list of content blocks from the model's response
    content_blocks = response["output"]["message"]["content"]

    reasoning = None
    text = None

    # Process each content block to find reasoning and response text
    for block in content_blocks:
        if "reasoningContent" in block:
            reasoning = block["reasoningContent"]["reasoningText"]["text"]
        if "text" in block:
            text = block["text"]

    return reasoning, text


if __name__ == "__main__":
    # Execute the example and display reasoning and final response
    result = reasoning_example()
    
    if result and result[0] is not None and result[1] is not None:
        reasoning, text = result
        
        print("\n" + "="*80)
        print("REASONING PROCESS (Thinking)")
        print("="*80)
        print(reasoning)
        
        print("\n" + "="*80)
        print("FINAL RESPONSE")
        print("="*80)
        print(text)
        print("="*80)
    else:
        print("\n[FAILED] Execution failed: Check error messages above.")


REASONING PROCESS (Thinking)
This question asks about enterprise business problems that can be solved by building AI agents with reasoning models. Let me explore several domains and applications where such agents could be valuable:

1. Customer Service and Support
   - Handling complex customer inquiries that require reasoning across multiple knowledge bases
   - Troubleshooting technical issues by reasoning through potential causes and solutions
   - Creating personalized recommendations based on customer history and preferences

2. Knowledge Management
   - Synthesizing information across large document repositories
   - Answering complex questions that require integrating multiple information sources
   - Maintaining and updating knowledge bases by identifying inconsistencies or gaps

3. Decision Support
   - Helping with complex business decisions by analyzing pros and cons
   - Risk assessment that requires evaluating multiple factors
   - Strategic planning that involves reasoni

# Strands Reasoning Example

This example demonstrates how to use Claude's reasoning capabilities through the Strands SDK with Amazon Bedrock models.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel

# Configure Claude 4 Sonnet with interleaved thinking
model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    additional_request_fields={
        "anthropic_beta": ["interleaved-thinking-2025-05-14"],
        "thinking": {"type": "enabled", "budget_tokens": 5000}
    }
)

# Create agent
agent = Agent(model=model)

# Ask a question that benefits from reasoning
response = agent("What are enterprise business problems that can be solved by building agents with reasoning models?")

# Extract and display results
thinking_content = None
final_answer = None

if hasattr(response, 'message') and isinstance(response.message, dict):
    for block in response.message.get('content', []):
        if isinstance(block, dict):
            # Extract reasoning content
            if 'reasoningContent' in block:
                reasoning = block['reasoningContent'].get('reasoningText', {})
                thinking_content = reasoning.get('text', '')
            # Extract final answer
            elif 'text' in block:
                final_answer = block['text']

# Display results
print("\n" + "="*80)
if thinking_content:
    print("🧠 REASONING PROCESS:")
    print("="*80)
    print(thinking_content)
    print("\n" + "="*80)

print("✅ FINAL ANSWER:")
print("="*80)
print(final_answer or "No answer found")
print("="*80)